This script is used to seperate images from one subject into two different regional groups: white matter and grey matter images. 
After the whole-slide scanning the produced 20x images generated are saved in a form LxxxCxxx.jpg. Lxxx stands for the Line (start from 0) and Cxxx stands for the Column (start from 0).

These individual 20x images are combined from the microscope software into one whole-slide image. This image is then manually handled using QuPath to generate a new whole-slide image with drawn ROI (red for grey matter, blue for white matter) based on the macro viewing of the tissue image.

Based on the "grid-like" way that the individual images are saved, we use the whole-slide image with the drawn ROI to automatically save them into two different folders for regional-based downstream analysis.

In [ ]:
#Install requirements
%pip install -r "../requirements.txt"

In [ ]:
#Import required libraries
import cv2
import numpy as np
import os
from scipy import ndimage

In [ ]:
# Load the whole slide image (adjust the input_path as needed)
whole_slide_image_path = 'input_path/whole_slide_image.jpg'
whole_slide_image = cv2.imread(whole_slide_image_path)

if whole_slide_image is None:
    raise FileNotFoundError(f"Error: Unable to load the image at {whole_slide_image_path}")

print(f"Whole slide image loaded successfully. Dimensions: {whole_slide_image.shape}")

# Define the size of individual images (tiles)
tile_width = 24  # Example size, adjust as needed
tile_height = 20 # Example size, adjust as needed

# Convert the image to HSV for color-based detection
hsv_image = cv2.cvtColor(whole_slide_image, cv2.COLOR_BGR2HSV)
print("Converted whole slide image to HSV color space.")

#Use blue and red masks to seperate the individual 20x images into different folders for downstream analysis
# Define the HSV range for the blue polygon (white matter)
lower_blue = np.array([100, 150, 0])
upper_blue = np.array([140, 255, 255])
blue_mask = cv2.inRange(hsv_image, lower_blue, upper_blue)

# Check if blue mask is generated
if np.count_nonzero(blue_mask) == 0:
    print("Warning: No blue color detected for white matter. Check the image and HSV range.")

# Define the HSV range for the red polygon (grey matter)
lower_red = np.array([0, 120, 70])
upper_red = np.array([10, 255, 255])
mask_red1 = cv2.inRange(hsv_image, lower_red, upper_red)

lower_red = np.array([170, 120, 70])
upper_red = np.array([180, 255, 255])
mask_red2 = cv2.inRange(hsv_image, lower_red, upper_red)
red_mask = mask_red1 | mask_red2

# Check if red mask is generated
if np.count_nonzero(red_mask) == 0:
    print("Warning: No red color detected for grey matter. Check the image and HSV range.")

print("Generated masks for blue (white matter) and red (grey matter) polygons.")

# Find contours for each mask
contours_blue, _ = cv2.findContours(blue_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours_red, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if not contours_blue:
    print("Warning: No contours found for white matter (blue polygons).")

if not contours_red:
    print("Warning: No contours found for grey matter (red polygons).")

print(f"Found {len(contours_blue)} contours for white matter and {len(contours_red)} contours for grey matter.")

contour_image = whole_slide_image.copy()
# Draw the contours for white matter (in blue)
cv2.drawContours(contour_image, contours_blue, -1, (255, 0, 0), 2)

# Draw the contours for grey matter (in red)
cv2.drawContours(contour_image, contours_red, -1, (0, 0, 255), 2)

# Save the image with contours drawn on it (adjust the output_path as needed)
output_contour_image_path = 'output_path/whole_slide_image_with_contours.jpg'
if cv2.imwrite(output_contour_image_path, contour_image):
    print(f"Image with contours saved as {output_contour_image_path}")
else:
    print(f"Error: Failed to save image with contours at {output_contour_image_path}")

# Calculate number of columns and rows
num_cols = whole_slide_image.shape[1] // tile_width
num_rows = whole_slide_image.shape[0] // tile_height

print(f"Grid will have {num_cols} columns and {num_rows} rows based on tile size ({tile_width}x{tile_height}).")

# Now proceed with creating the grid
grid_image = whole_slide_image.copy()

#Detect the images belonging to each region based on their name (LxxxCxxx) in which Lxxx stands for specific Line and Cxxx stands for specific column
# Draw the grid
for i in range(num_cols):
    for j in range(num_rows):
        top_left_x = i * tile_width
        top_left_y = j * tile_height
        bottom_right_x = (i + 1) * tile_width
        bottom_right_y = (j + 1) * tile_height
        cv2.rectangle(grid_image, (top_left_x, top_left_y), (bottom_right_x, bottom_right_y), (0, 255, 0), 1)

# Save the image with the grid overlaid (adjust the output_path as needed)
output_grid_image_path = 'output_path/whole_slide_image_with_grid.jpg'
if cv2.imwrite(output_grid_image_path, grid_image):
    print(f"Image with grid saved as {output_grid_image_path}")
else:
    print(f"Error: Failed to save image with grid at {output_grid_image_path}")

# Initialize lists to store (L, C) indices for each tissue type
white_matter_indices = []
grey_matter_indices = []

# Loop over each grid cell and determine if it belongs to white or grey matter
for i in range(num_cols):
    for j in range(num_rows):
        x_start = i * tile_width
        y_start = j * tile_height
        cell_center = (x_start + tile_width // 2, y_start + tile_height // 2)

        # Check if the cell center is inside the blue polygon (white matter)
        if any(cv2.pointPolygonTest(contour, cell_center, False) >= 0 for contour in contours_blue):
            white_matter_indices.append((j, i))  # j for row (L), i for column (C)

        # Check if the cell center is inside the red polygon (grey matter)
        elif any(cv2.pointPolygonTest(contour, cell_center, False) >= 0 for contour in contours_red):
            grey_matter_indices.append((j, i))  # j for row (L), i for column (C)
    
# Safety check: Print number of indices found for each tissue type
print(f"Identified {len(white_matter_indices)} grid cells for white matter (blue).")
print(f"Identified {len(grey_matter_indices)} grid cells for grey matter (red).")

# Save the indices to separate files (adjust the output_path and sample_name_folder as needed)
output_dir = 'output_path/sample_name_folder'
os.makedirs(output_dir, exist_ok=True)
print(f"Created output directory: {output_dir}")

with open(os.path.join(output_dir, 'white_matter_indices.txt'), 'w') as f:
    for (L, C) in white_matter_indices:
        f.write(f'L{L:03d}C{C:03d}\n')
print("White matter indices saved to white_matter_indices.txt")

with open(os.path.join(output_dir, 'grey_matter_indices.txt'), 'w') as f:
    for (L, C) in grey_matter_indices:
        f.write(f'L{L:03d}C{C:03d}\n')
print("Grey matter indices saved to grey_matter_indices.txt")

print("Grid labeling complete. Indices saved to the 'polygon_indices' directory.")


In [ ]:
# Path to the directory where individual images are stored (adjust the individual_images_path for the studied sample as needed)
individual_images_dir = 'individual_images_path'
if not os.path.exists(individual_images_dir):
    raise FileNotFoundError(f"Error: The directory {individual_images_dir} does not exist.")

print(f"Directory for individual images: {individual_images_dir}")

# Directory where the (L, C) index files are stored (adjust the output_path and sample_name_folder as needed)
polygon_indices_dir = 'output_path/sample_name_folder'
if not os.path.exists(polygon_indices_dir):
    raise FileNotFoundError(f"Error: The directory {polygon_indices_dir} does not exist.")

print(f"Directory for polygon indices: {polygon_indices_dir}")

# Create output directories for white matter and grey matter (adjust the output_path as needed)
white_matter_dir = 'output_path/white_matter'
grey_matter_dir = 'output_path/grey_matter'
os.makedirs(white_matter_dir, exist_ok=True)
os.makedirs(grey_matter_dir, exist_ok=True)

print(f"Created output directories: '{white_matter_dir}' and '{grey_matter_dir}'")

# Function to load indices from a file
def load_indices(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Error: The file {file_path} does not exist.")
    with open(file_path, 'r') as f:
        return [line.strip() for line in f]

# Load indices for white and grey matter
white_matter_indices = load_indices(os.path.join(polygon_indices_dir, 'white_matter_indices.txt'))
grey_matter_indices = load_indices(os.path.join(polygon_indices_dir, 'grey_matter_indices.txt'))

print(f"Loaded {len(white_matter_indices)} white matter indices and {len(grey_matter_indices)} grey matter indices.")

# Move the corresponding images to the appropriate output directories
for index in white_matter_indices:
    tile_name = f'Scan_{index}.jpg'
    tile_path = os.path.join(individual_images_dir, tile_name)
    
    if not os.path.exists(tile_path):
        print(f"Warning: Image {tile_name} does not exist in {individual_images_dir}.")
    else:
        cv2.imwrite(os.path.join(white_matter_dir, tile_name), cv2.imread(tile_path))

print(f"White matter images have been saved to {white_matter_dir}")

for index in grey_matter_indices:
    tile_name = f'Scan_{index}.jpg'
    tile_path = os.path.join(individual_images_dir, tile_name)
    
    if not os.path.exists(tile_path):
        print(f"Warning: Image {tile_name} does not exist in {individual_images_dir}.")
    else:
        cv2.imwrite(os.path.join(grey_matter_dir, tile_name), cv2.imread(tile_path))

print(f"Grey matter images have been saved to {grey_matter_dir}")
print("Image separation complete. Check the 'output_white_matter' and 'output_grey_matter' directories for results.")